# 12.10 模型有聽聲音嗎？


一個問句總是「這是高音還是低音」，資料裡高音佔九成，模型只答高也可能得九成。要證明它聽了波形，需要換聲音、保持問句相同，觀察答案是否按頻率規則改變。本節先算完全不聽的基準，讓高分有可比較的起點。

前置是[12.9聲音訓練接口](https://birdhackor.github.io/tiny-perceptron-vlm/12.9.html)與[11.8替圖檢驗](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)。規則在這裡明確設為頻率大於300Hz叫high，否則low；不是宣稱人類聽覺的統一高低音界線。保留測試應含沒用於訓練的頻率，但仍按相同規則給真值。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
frequencies = [200, 220, 440, 660]
truth = ["high" if f > 300 else "low" for f in frequencies]
no_audio = ["high"] * len(frequencies)
accuracy = sum(a == b for a, b in zip(truth, no_audio)) / len(truth)
print("頻率", frequencies)
print("標準答案", truth)
print("不聽聲音的固定猜測", accuracy)

輸入四個Hz值，兩低兩高。清單生成式逐個判斷f>300，輸出[low,low,high,high]。`no_audio`故意不讀取波形、全猜high，後面逐項核對只有兩題正確，輸出0.5。這不是訓練後音訊模型的測量，只有構造的規則基準。

真實模型測試應先保存原聲音的回答，再將200Hz換成440Hz，仍問同一句，正確答案必須從low變high。替換660為440，兩者同屬high，理想類別答案則不必變化。這種按規則的變化比「兩個輸出不一樣」更嚴格；隨機數字也會不同，卻不一定答對。

另可把錄音打亂配對，但要分開兩種評分。依換入頻率重新標答案，量的是新波形的正確率，真正聽聲音的模型仍應答對；保留原答案則量原答案一致率，是檢查輸出是否依聲音改變，不是新波形的真實正確率。例如四段從[200,220,440,660]换成[440,660,200,220]，新答案為[high,high,low,low]；理想模型照新聲音回答，按新標籤是1.0，按舊標籤則是0.0。不能把前者下降當成成功，也不能把後者下降直接稱為辨音能力退步。靜音在規則裡沒有有效單音頻率，目標不應直接繼承原標籤；應該作為缺資訊的單獨測試，看模型是否仍固定猜high。單音識別成功也不能外推成理解說話內容，真實語音的噪聲、多人聲音與詞句需要額外任務。

高低音真值來自合成頻率，不是來自模型輸出。因此替換聲音後必須同步重新計算真值，而不能保留原答案low來測440Hz。另一方面，問題字串不應偷偷加「這是高音」，否則文字已經泄漏目標，聽不聽都能答。一個好的成對實驗只換實際波形與由規則確定的標籤，其他可見條件固定。將問句-only、原聲音、替換聲音與靜音結果並排，會讓每項證據比較清楚；只有原聲音一欄高分時，應再查其它欄，避免靠單一成功故事下結論。

練習只把測試頻率改成[180,260,480,720]，先預測標籤仍兩低兩高、固定猜測仍0.5，再執行核對。它們是不同數值而同一規則的例子；若模型只背過200與440，新頻率測試更容易暴露這種限制。
